<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 13.2: Tool-Calling Investigation Agent-Answers
## Adding Security Tools to Your Agent

**Estimated Time**: 45-60 minutes

### Learning Objectives
- Create security investigation tools using the `@tool` decorator
- Bind tools to an LLM and understand how it decides when to call them
- Build a ReAct (Reason + Act) agent loop using LangGraph
- Use message-based state to track the full conversation between LLM and tools

### What You'll Build
An agent that autonomously investigates emails by calling security tools before making a classification:
```
START → investigator → should_continue? →(tools)→ execute_tools → investigator
                                        →(done)→ END
```

### Starter Code from Lab 1
This lab provides prior components so you can start fresh even if Lab 1 isn't complete.

In [ ]:
# Cell 2: Environment setup
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file
load_dotenv(find_dotenv())

# Verify the API key is set
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found! Create a .env file with your key."
print("Environment loaded successfully.")

In [ ]:
# Cell 3: Imports
import json
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict

import sys
sys.path.insert(0, "..")
from utils.email_loader import load_emails, format_email_for_analysis
from utils.mock_tools import (
    check_url_reputation,
    check_sender_reputation,
    check_email_authentication,
    SECURITY_TOOLS,
)
from utils.display_helpers import display_graph_mermaid

print("All imports successful.")

In [ ]:
# Cell 4: Load email data
emails = load_emails(small=False)
print(f"Loaded {len(emails)} emails.\n")

# Pick a few interesting emails for demonstration:
#   emails[0]  -> EMAIL-001: credential-harvesting phishing
#   emails[5]  -> EMAIL-028: legitimate code review
#   emails[25] -> EMAIL-021: vishing (fake Norton renewal)
#   emails[47] -> EMAIL-048: suspicious (security newsletter with urgency)
test_emails = [emails[0], emails[5], emails[25], emails[47]]
for e in test_emails:
    print(f"{e['id']}: {e['subject'][:60]}... ({e['ground_truth']})")

## Concept: Message-Based State

In Lab 1, our state was a simple `TypedDict` with fields like `email`, `classification`, and `confidence`. That works for a straight-line pipeline, but a **tool-calling agent** needs something richer: a **message history**.

Why messages?
- The LLM sends a message requesting a tool call ("I'd like to check this URL").
- The tool runs and its result is appended as a **tool message**.
- The LLM sees the full conversation, reasons about the result, and may call another tool or give its final answer.

LangGraph provides `add_messages` -- a **reducer** that ensures new messages are *appended* to the list rather than replacing it. This is the key difference from a plain `list` type annotation.

```python
from langgraph.graph.message import add_messages
from typing import Annotated

class MyState(TypedDict):
    messages: Annotated[list, add_messages]  # append-only message list
```

In [ ]:
# Cell 6: Define state

class InvestigationState(TypedDict):
    """State for the investigation agent.
    
    Messages track the full LLM <-> tool conversation.
    The add_messages reducer ensures new messages are appended,
    not replaced.
    """
    # TODO: define a single 'messages' field using the add_messages reducer.
    # Hint: Annotated[list, add_messages]
    messages: ____

print("InvestigationState defined.")

## Concept: Security Tools

Our agent has access to three security investigation tools, already implemented in `utils/mock_tools.py`. Each is decorated with LangChain's `@tool` decorator, which:

1. Registers the function as a **tool** the LLM can call.
2. Uses the **docstring** to tell the LLM what the tool does and when to use it.
3. Automatically generates an **args schema** from the function signature so the LLM knows what arguments to provide.

The three tools are:

| Tool | Purpose |
|------|--------|
| `check_url_reputation` | Looks up a URL against a threat intelligence database |
| `check_sender_reputation` | Checks if a sender's email/domain is known-malicious or trusted |
| `check_email_authentication` | Evaluates SPF, DKIM, and DMARC headers for spoofing indicators |

These tools simulate real security APIs (VirusTotal, email gateways, etc.) using a local JSON database so you don't need paid API keys.

In [ ]:
# Cell 8: Explore the tools -- see what the LLM will know about each one
for tool in SECURITY_TOOLS:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Schema: {json.dumps(tool.args_schema.model_json_schema(), indent=2)}")
    print()

In [ ]:
# Cell 9: Quick tool test -- call the tools directly to see their output
print("=== URL Reputation Check ===")
print(check_url_reputation.invoke({"url": "https://micros0ft-verify.com/secure/login?id=8f3a2b"}))
print()

print("=== Sender Reputation Check ===")
print(check_sender_reputation.invoke({"email_address": "security-alert@micros0ft-verify.com"}))
print()

print("=== Email Authentication Check ===")
print(check_email_authentication.invoke({
    "from_address": "security-alert@micros0ft-verify.com",
    "spf": "fail",
    "dkim": "fail",
    "dmarc": "fail",
}))

## Concept: Binding Tools to the LLM

Modern LLMs like Claude support **function calling** (also called tool calling). When we call `llm.bind_tools(tools)`, we tell the LLM:

1. **What tools are available** -- their names, descriptions, and parameter schemas.
2. **How to call them** -- the LLM includes structured `tool_calls` in its response when it wants to use a tool.

The LLM does NOT execute the tool itself. It simply says "I'd like to call `check_url_reputation` with URL X". The **ToolNode** (which we'll add shortly) actually runs the tool and returns the result.

```python
llm_with_tools = llm.bind_tools(tools)
# Now when we call llm_with_tools.invoke(messages),
# the response may contain tool_calls instead of (or in addition to) text.
```

In [ ]:
# Cell 11: Initialize LLM with tools
llm = ChatAnthropic(model="claude-opus-4-8", max_tokens=4096)
# TODO: give the LLM access to the security tools.
# Hint: llm.bind_tools(SECURITY_TOOLS)
llm_with_tools = ____

print(f"LLM initialized: {llm.model}")
print(f"Tools bound: {[t.name for t in SECURITY_TOOLS]}")

## Concept: The Investigator Node

The **investigator** is the central node in our graph. Each time it runs, it:

1. Takes the current message history from state.
2. Ensures a **system prompt** is present (telling the LLM how to behave).
3. Sends the messages to the LLM.
4. Returns the LLM's response (which gets appended to the message list).

The LLM's response will be one of two things:
- **Tool calls**: The LLM wants to gather more evidence (e.g., check a URL). The response's `tool_calls` field will be populated.
- **Text response**: The LLM has gathered enough evidence and is providing its final verdict. The `content` field will contain the analysis.

In [ ]:
# Cell 13: Define the investigator node

SYSTEM_PROMPT = """You are an expert email security analyst investigating a potentially malicious email.

You have access to security tools to help your investigation. Use them to gather evidence before making your final assessment.

Investigation protocol:
1. First, analyze the email content for obvious red flags
2. Check any URLs in the email against threat intelligence
3. Verify the sender's reputation
4. Check the email authentication headers (SPF, DKIM, DMARC)
5. Based on ALL evidence gathered, provide your final verdict

When you have gathered enough evidence, provide your final assessment with:
- Verdict: phishing, suspicious, or legitimate
- Confidence: a percentage (e.g., 95%)
- Key findings from your investigation
- Recommended action (block, quarantine, or allow)

Be thorough - always use your tools before making a final decision."""


def investigator(state: InvestigationState) -> dict:
    """The investigator node: sends messages to the LLM which may call tools or give a final answer."""
    messages = state["messages"]
    # Ensure system prompt is first
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(messages)

    # TODO: invoke the tool-bound LLM on the messages.
    response = ____
    # TODO: return the response wrapped so add_messages appends it.
    # Hint: {"messages": [response]}
    return ____


print("Investigator node defined.")

## Concept: ToolNode and Conditional Routing

Two more pieces complete the ReAct loop:

### ToolNode
`ToolNode` is a LangGraph prebuilt that automatically:
- Reads the `tool_calls` from the last AI message
- Executes each requested tool
- Returns the results as `ToolMessage` objects (appended to state)

### Conditional Routing (`should_continue`)
After the investigator runs, we need to decide:
- **If the LLM requested tool calls** -> route to `execute_tools` (the ToolNode)
- **If the LLM gave a final text answer** -> route to `END`

We check this by looking at the `tool_calls` attribute on the last message.

In [ ]:
# Cell 15: Define routing function and ToolNode

# TODO: create a ToolNode that can execute the SECURITY_TOOLS.
tool_node = ____


def should_continue(state: InvestigationState) -> str:
    """Route based on whether the LLM wants to call more tools."""
    last_message = state["messages"][-1]
    # If the LLM made tool calls, route to execute_tools
    # TODO: if the LLM made tool calls, route to "execute_tools"; else END.
    # Hint: check last_message.tool_calls
    if ____:
        return "execute_tools"
    return END


print("ToolNode and routing function defined.")

## Concept: Building the ReAct Graph

The **ReAct** (Reason + Act) pattern is a loop:

1. **Reason**: The investigator (LLM) looks at the evidence so far and decides what to do.
2. **Act**: If the LLM wants more info, it calls a tool. The tool result is added to the conversation.
3. **Repeat**: The LLM sees the tool result, reasons again, and may call another tool or finalize.

```
START -> investigator -> [should_continue?]
                             |
                    tool_calls? YES -> execute_tools -> investigator (loop back)
                             |
                    tool_calls? NO  -> END
```

The agent keeps looping until the LLM decides it has enough evidence and provides a final text response.

In [ ]:
# Cell 17: Build the graph

workflow = StateGraph(InvestigationState)

# Add nodes
workflow.add_node("investigator", investigator)
workflow.add_node("execute_tools", tool_node)

# Add edges
workflow.add_edge(START, "investigator")
# TODO: route from investigator using should_continue, then loop tools back.
workflow.add_conditional_edges(
    "investigator",
    ____,
    {"execute_tools": "execute_tools", END: END},
)
workflow.add_edge(____, ____)

# Compile
graph = workflow.compile()
print("Investigation agent compiled!")

In [ ]:
# Cell 18: Visualize the graph
display_graph_mermaid(graph, "Lab 2: Investigation Agent (ReAct Loop)")

## Run the Investigation

Now let's see the agent in action. We'll start with a single phishing email to observe the full tool-calling conversation, then run all four test emails.

In [ ]:
# Cell 20: Investigate a single phishing email (credential harvesting)
email = test_emails[0]
print(f"Investigating: {email['id']} - {email['subject']}")
print(f"Ground truth: {email['ground_truth']}")
print("=" * 60)

email_text = format_email_for_analysis(email)

result = graph.invoke({
    "messages": [HumanMessage(content=f"Investigate this email:\n\n{email_text}")]
})

# Print the full conversation to see tool usage
for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n>>> LLM requested tools: {[tc['name'] for tc in msg.tool_calls]}")
    elif msg.type == "tool":
        print(f"\n<<< Tool result ({msg.name}):\n{msg.content[:200]}...")
    elif msg.type == "ai" and msg.content:
        print(f"\n--- Final Analysis ---\n{msg.content}")
    elif msg.type == "human":
        print(f"\n[Input]: {msg.content[:100]}...")

In [ ]:
# Cell 21: Investigate all test emails and compare with ground truth
results = []
for email in test_emails:
    email_text = format_email_for_analysis(email)
    result = graph.invoke({
        "messages": [HumanMessage(content=f"Investigate this email:\n\n{email_text}")]
    })

    # Get the final message
    final_msg = result["messages"][-1].content

    # Count tool calls made during investigation
    tool_calls = sum(
        len(msg.tool_calls)
        for msg in result["messages"]
        if hasattr(msg, "tool_calls") and msg.tool_calls
    )

    print(f"\n{'=' * 60}")
    print(f"Email: {email['id']} - {email['subject'][:50]}")
    print(f"Ground truth: {email['ground_truth']}")
    print(f"Tools called: {tool_calls}")
    print(f"Final analysis:\n{final_msg[:500]}")

    results.append({
        "email_id": email["id"],
        "ground_truth": email["ground_truth"],
        "tool_calls": tool_calls,
        "final_analysis": final_msg,
    })

## Checkpoint Checklist

- [ ] Graph compiles with the ReAct loop structure
- [ ] Mermaid diagram shows: investigator -> execute_tools loop with conditional edge
- [ ] LLM autonomously calls security tools (URL check, sender check, auth check)
- [ ] Agent makes multiple tool calls before reaching a final verdict
- [ ] Phishing emails are identified with tool-based evidence
- [ ] Legitimate emails pass tool checks and are correctly classified

## What's Next?

In **Lab 3**, we'll add conditional routing to create tiered analysis -- quick triage routes emails to different processing paths based on risk level. High-risk emails get full tool-based investigation, while obviously safe emails get fast-tracked.